In [ ]:
import faiss
import numpy as np
import pickle
truth = pickle.load(open('/data3/menghaotian/Traj_sim/ConvTraj/data/0_geolife/dtw_test_distance_matrix_result', "rb"))
def construct_gt_from_matrix(dist_matrix):
    """
    从预计算的距离矩阵中构建 Top-1 Ground Truth
    
    :param dist_matrix: 形状为 (nq, nb) 的 NumPy 数组
    :return: D_gt (距离), I_gt (索引)，形状均为 (nq, 1)
    """
    print(f"正在处理距离矩阵，形状: {dist_matrix.shape}...")
    
    # 1. 获取索引 (I_gt)
    # axis=1 表示在每一行中横向比较，找到最小值的下标
    # argmin 比 argsort 快很多，因为我们只需要 Top-1
    min_indices = np.argmin(dist_matrix, axis=1)
    
    # 调整形状为 (nq, 1) 以匹配 Faiss 的标准输出格式
    I_gt = min_indices.reshape(-1, 1)
    
    # 2. 获取距离 (D_gt)
    # 使用高级索引提取对应的距离值
    # row_indices 是 [0, 1, 2, ..., nq-1]
    row_indices = np.arange(dist_matrix.shape[0])
    min_distances = dist_matrix[row_indices, min_indices]
    
    # 同样调整形状为 (nq, 1)
    D_gt = min_distances.reshape(-1, 1)
    
    return D_gt, I_gt
def evaluate_metrics(I_pred, I_gt, k_eval_list):
    """
    计算不同 K 值的 Recall/Hit Rate
    :param I_pred: 预测的索引矩阵 (nq, k_search)
    :param I_gt: 真实的最近邻索引 (nq, 1) - 这里假设只找 Top-1 Ground Truth
    :param k_eval_list: 需要评估的 K 值列表，例如 [10, 50]
    """
    nq = I_pred.shape[0]
    results = {}

    for k in k_eval_list:
        # 截取前 k 个预测结果
        I_sub = I_pred[:, :k]
        
        # 检查 GT 是否在预测的这 k 个结果中
        # any(axis=1) 会返回一个布尔数组，表示每个 query 是否命中
        hits = np.any(I_sub == I_gt, axis=1)
        
        # 计算比例
        recall = np.sum(hits) / nq
        results[f"Recall@{k}"] = recall
        results[f"HR@{k}"] = recall # 在 Top-1 任务中，HR 和 Recall 数值一样
        
    return results

# ==========================================
# 1. 准备数据 (模拟 SIFT1M 规模)
# ==========================================
d = 128           # 向量维度

xt = np.load('/data3/menghaotian/Traj_sim/PDT-VQ/data/convTraj/train_dtw_feature_test',allow_pickle=True)  # 训练集
xb = np.load('/data3/menghaotian/Traj_sim/PDT-VQ/data/convTraj/base_dtw_feature_test',allow_pickle=True)   # 库
xq = np.load('/data3/menghaotian/Traj_sim/PDT-VQ/data/convTraj/query_dtw_feature_test',allow_pickle=True)  # 查询
'''
np.random.seed(1234)
xb = np.random.random((nb, d)).astype('float32')
xq = np.random.random((nq, d)).astype('float32')
xt = np.random.random((nt, d)).astype('float32')
'''
# ==========================================
# 2. 计算 Ground Truth (暴力搜索)
# ==========================================
print("正在计算 Ground Truth (Brute Force)...")
D_gt, I_gt = construct_gt_from_matrix(truth)
'''
index_gt = faiss.IndexFlatL2(d)
index_gt.add(xb)
# 我们只关心 Top-1 的真实邻居
D_gt, I_gt = index_gt.search(xq, 1) 
'''
# ==========================================
# 3. 构建 Faiss VQ 索引 (IVFPQ)
# ==========================================
# 参数设置
nlist = 100       # 聚类中心数 (Coarse quantizer centroids)
m = 8             # PQ 切分的子空间数量 (Sub-quantizers)
nbits = 8         # 每个子向量的比特数 (通常是 8)

print("正在构建 IVFPQ 索引...")
quantizer = faiss.IndexFlatL2(d)  # 粗量化器
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)

# 训练索引
index.train(xt)
# 添加数据
index.add(xb)

# ==========================================
# 4. 搜索与评估
# ==========================================
index.nprobe = 10  # 搜索探针数，越大越准但越慢

# 设定搜索深度 L=50 (即 Recall 10 @ 50 中的 50)
# 这意味着我们从索引中取回 50 个候选者
search_L = 50 
print(f"正在执行搜索 (候选列表大小 L={search_L})...")
D_pred, I_pred = index.search(xq, search_L)

# ==========================================
# 5. 计算指标
# ==========================================
# 我们需要评估 HR@10 和 HR@50
# HR@50 等同于 Recall@50
# HR@10 等同于 Recall@10 (即 Recall 10 @ 50 的前半部分含义)

metrics = evaluate_metrics(I_pred, I_gt, k_eval_list=[10, 50])

print("\n=== 评估结果 ===")
print(f"Recall@10 (HR@10): {metrics['Recall@10']:.4f}")
print(f"Recall@50 (HR@50): {metrics['Recall@50']:.4f}")

# 解释 Recall 10 @ 50
print(f"\n指标解释:")
print(f"- Recall@50 表示候选列表(L=50)的召回率上限。")
print(f"- Recall@10 表示如果我们不重排序，直接取前10个的准确率。")
print(f"- 如果你有重排序模型(Re-ranker)，你会对这50个结果重排，然后重新计算Recall@10。")

正在计算 Ground Truth (Brute Force)...
正在处理距离矩阵，形状: (1000, 9386)...
正在构建 IVFPQ 索引...
正在执行搜索 (候选列表大小 L=50)...

=== 评估结果 ===
Recall@10 (HR@10): 0.6480
Recall@50 (HR@50): 0.8920

指标解释:
- Recall@50 表示候选列表(L=50)的召回率上限。
- Recall@10 表示如果我们不重排序，直接取前10个的准确率。
- 如果你有重排序模型(Re-ranker)，你会对这50个结果重排，然后重新计算Recall@10。


WARNING clustering 3000 points to 100 centroids: please provide at least 3900 training points
WARNING clustering 3000 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3000 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3000 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3000 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3000 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3000 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3000 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3000 points to 256 centroids: please provide at least 9984 training points


In [11]:
import numpy as np

def compute_l2_distance_matrix(xq, xb):
    """
    使用 numpy 高效计算欧氏距离矩阵 (L2 squared)
    利用公式: (a-b)^2 = a^2 + b^2 - 2ab
    """
    print(f"正在计算 L2 距离矩阵: Query{xq.shape} vs Base{xb.shape} ...")
    
    # 1. 计算 query 的平方和 (nq, 1)
    q_norm = np.sum(xq**2, axis=1).reshape(-1, 1)
    
    # 2. 计算 base 的平方和 (1, nb)
    b_norm = np.sum(xb**2, axis=1).reshape(1, -1)
    
    # 3. 计算 2ab (内积)
    # 注意：如果数据量非常大，dot 操作可能会内存溢出，需要分块处理
    dot_prod = np.dot(xq, xb.T)
    
    # 4. 组合公式
    # 利用广播机制: (nq, 1) + (1, nb) - (nq, nb)
    dist_matrix = q_norm + b_norm - 2 * dot_prod
    
    return dist_matrix

def get_topk_indices(dist_matrix, k=50):
    """
    从距离矩阵中获取每一行最小的 k 个索引
    """
    # np.argpartition 比全排序(argsort)快很多
    # 它保证前 k 个元素是最小的，但前 k 个内部不一定有序
    unsorted_topk_indices = np.argpartition(dist_matrix, k, axis=1)[:, :k]
    
    # 取出这 k 个对应的距离值
    row_indices = np.arange(dist_matrix.shape[0])[:, None]
    topk_dists = dist_matrix[row_indices, unsorted_topk_indices]
    
    # 对这 k 个进行局部排序 (为了准确计算 HR@10)
    sorted_order = np.argsort(topk_dists, axis=1)
    
    # 得到最终有序的 top-k 索引
    final_indices = unsorted_topk_indices[row_indices, sorted_order]
    
    return final_indices

def calculate_hit_rate(pred_indices, gt_indices, k_list=[10, 50]):
    """
    计算 HR@K (Recall@K)
    pred_indices: (nq, max_k)
    gt_indices: (nq, 1) - Top-1 Ground Truth
    """
    nq = pred_indices.shape[0]
    results = {}
    
    for k in k_list:
        # 截取前 k 个预测
        preds_k = pred_indices[:, :k]
        
        # 检查 GT 是否在预测行中
        # any(axis=1) 返回 shape (nq,) 的布尔数组
        hits = np.any(preds_k == gt_indices, axis=1)
        
        # 计算平均值
        hr = np.mean(hits)
        results[f"HR@{k}"] = hr
        
    return results

# ==========================================
# 1. 数据准备 (模拟)
# ==========================================

dim = 128     # 向量维度


xb = np.load('/data3/menghaotian/Traj_sim/PDT-VQ/data/convTraj/base_dtw_feature_test',allow_pickle=True)   # 库
xq = np.load('/data3/menghaotian/Traj_sim/PDT-VQ/data/convTraj/query_dtw_feature_test',allow_pickle=True)  # 查询
nq = xq.shape[0]
nb = xb.shape[0]    # 数据库数量
# 模拟预先算好的 DTW 矩阵 (用于生成 GT)
# 实际情况中，这是你已经有的矩阵
dtw_matrix = pickle.load(open('/data3/menghaotian/Traj_sim/ConvTraj/data/0_geolife/dtw_test_distance_matrix_result', "rb"))

# ==========================================
# 2. 核心流程
# ==========================================

# A. 获取 Ground Truth (来自 DTW)
# axis=1 找每行最小值的索引，reshape成 (nq, 1)
gt_indices = np.argmin(dtw_matrix, axis=1).reshape(-1, 1)

# B. 获取预测结果 (基于原始向量的 L2 距离)
# 计算 L2 距离
l2_dist_matrix = compute_l2_distance_matrix(xq, xb)

# 提取前 50 个最近邻 (预测的索引)
pred_indices_top50 = get_topk_indices(l2_dist_matrix, k=50)

# C. 计算评估指标
metrics = calculate_hit_rate(pred_indices_top50, gt_indices, k_list=[10, 50])

print("\n=== 评估结果 ===")
print(f"对比方式: 预测(L2 Raw) vs 真值(DTW)")
print(f"HR@10 (Recall@10): {metrics['HR@10']:.4f}")
print(f"HR@50 (Recall@50): {metrics['HR@50']:.4f}")

# 结果解释
print("\n[解释]")
print("HR@10:  表示用欧氏距离搜出来的前10个结果里，包含DTW最近邻的概率。")
print("HR@50:  表示用欧氏距离搜出来的前50个结果里，包含DTW最近邻的概率。")

正在计算 L2 距离矩阵: Query(1000, 128) vs Base(9386, 128) ...

=== 评估结果 ===
对比方式: 预测(L2 Raw) vs 真值(DTW)
HR@10 (Recall@10): 0.6780
HR@50 (Recall@50): 0.9070

[解释]
HR@10:  表示用欧氏距离搜出来的前10个结果里，包含DTW最近邻的概率。
HR@50:  表示用欧氏距离搜出来的前50个结果里，包含DTW最近邻的概率。


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
xt = np.load('/data3/menghaotian/Traj_sim/PDT-VQ/data/convTraj/train_dtw_feature_test',allow_pickle=True)  # 训练集
xb = np.load('/data3/menghaotian/Traj_sim/PDT-VQ/data/convTraj/base_dtw_feature_test',allow_pickle=True)   # 库
xq = np.load('/data3/menghaotian/Traj_sim/PDT-VQ/data/convTraj/query_dtw_feature_test',allow_pickle=True)  # 查询

class DPQ(nn.Module):
    def __init__(self, input_dim, num_subspaces, num_centroids):
        """
        Differentiable Product Quantization 模型
        
        Args:
            input_dim (int): 输入向量维度 (例如 128)
            num_subspaces (int): 切分成多少个子空间 M (例如 4)
            num_centroids (int): 每个子空间的聚类中心数 K (例如 256)
        """
        super(DPQ, self).__init__()
        
        assert input_dim % num_subspaces == 0, "输入维度必须能被子空间数整除"
        
        self.d_sub = input_dim // num_subspaces
        self.m = num_subspaces
        self.k = num_centroids
        
        # 定义码本 (Codebook)
        # 形状: [M, K, D/M] -> [子空间数, 中心点数, 子维度]
        # 这是一个可学习的参数矩阵
        self.codewords = nn.Parameter(torch.randn(self.m, self.k, self.d_sub))

    def compute_distances(self, x):
        """
        计算输入 x 的每个子段与对应子空间所有中心点的距离
        """
        # x shape: [Batch, D]
        batch_size = x.shape[0]
        
        # 1. 将 x 切分为 M 个子向量
        # shape: [Batch, M, D/M]
        x_reshaped = x.view(batch_size, self.m, self.d_sub)
        
        # 2. 扩展维度以便广播计算距离
        # x_expanded: [Batch, M, 1, D/M]
        # code_expanded: [1, M, K, D/M]
        x_expanded = x_reshaped.unsqueeze(2)
        code_expanded = self.codewords.unsqueeze(0)
        
        # 3. 计算欧氏距离平方
        # shape: [Batch, M, K]
        dist = torch.sum((x_expanded - code_expanded) ** 2, dim=-1)
        return dist, x_reshaped

    def forward(self, x, tau=1.0, hard=False):
        """
        Args:
            x: 输入向量
            tau: Softmax 温度系数，越小越接近 Hard Assignment
            hard: 是否在 Forward Pass 中使用 Hard 量化（但在 Backward 中传导梯度）
        """
        # dist shape: [Batch, M, K]
        dist, x_reshaped = self.compute_distances(x)
        
        # === 核心：软分配 (Soft Assignment) ===
        # 使用负距离进行 Softmax，距离越小，概率越大
        # probs shape: [Batch, M, K]
        probs = F.softmax(-dist / tau, dim=-1)
        
        if hard:
            # 推理阶段或 Hard 训练策略：直接选概率最大的 (One-hot)
            # 使用 Straight-Through Estimator (STE) 技巧
            # 这种写法保证了前向传播是 Hard 的，反向传播利用 Soft 的梯度
            index = probs.argmax(dim=-1)
            hard_probs = F.one_hot(index, num_classes=self.k).float()
            # STE: value = hard, gradient = soft
            probs = (hard_probs - probs).detach() + probs

        # === 重构 ===
        # 利用概率加权求和中心点
        # probs: [Batch, M, K]
        # codewords: [M, K, D_sub]
        # 这里的矩阵乘法需要在 M 维度内进行
        
        # [Batch, M, K] @ [Batch, M, K, D_sub] -> 太麻烦，用 einsum 最清晰
        # b=batch, m=subspaces, k=centroids, d=sub_dim
        x_quantized = torch.einsum('bmk,mkd->bmd', probs, self.codewords)
        
        # 拼回原始维度 [Batch, D]
        x_out = x_quantized.reshape(x.shape[0], -1)
        
        return x_out, probs, dist

def loss_function(x_original, x_recon, probs):
    """
    DPQ 损失函数
    """
    # 1. MSE 重构损失
    mse_loss = F.mse_loss(x_recon, x_original)
    
    # 2. 熵正则化 (Diversity Loss)
    # 我们希望在一个 Batch 内，所有中心点被选中的概率是均衡的
    # 计算每个中心点在 Batch 内的平均被选中概率
    # avg_probs shape: [M, K]
    avg_probs = torch.mean(probs, dim=0) 
    
    # 最大化熵 = 最小化负熵
    # Entropy = - sum(p * log(p))
    entropy = -torch.sum(avg_probs * torch.log(avg_probs + 1e-10))
    
    # 我们希望熵最大 (接近均匀分布)，所以 loss 要减去熵
    # 或者理解为：最小化 (最大熵 - 当前熵)
    # 这里简单地作为惩罚项
    entropy_loss = -entropy
    
    return mse_loss, entropy_loss

# ==========================================
# 训练演示
# ==========================================

# 1. 设置参数
D = 128   # 向量维度
M = 4     # 切成4段 (每段32维)
K = 256   # 码本大小 (8 bit)
BATCH_SIZE = 64
LR = 0.001
EPOCHS = 1000

# 2. 初始化模型和数据
model = DPQ(input_dim=D, num_subspaces=M, num_centroids=K)
optimizer = optim.Adam(model.parameters(), lr=LR)

# 模拟数据 (例如 SIFT 特征)
train_data = xt # 随机高斯分布数据

print(f"开始训练 DPQ... 维度:{D}, 子空间:{M}, 码本:{K}")

for epoch in range(EPOCHS):
    # 随机采样 Batch
    indices = torch.randperm(train_data.size(0))[:BATCH_SIZE]
    batch_x = train_data[indices]
    
    # 动态调整温度 (可选)：训练初期温度高(更软)，后期温度低(更硬)
    # tau = max(0.1, 1.0 - epoch / EPOCHS)
    tau = 1.0 
    
    optimizer.zero_grad()
    
    # 前向传播
    # 这里演示 Soft 模式，方便梯度传播
    x_recon, probs, _ = model(batch_x, tau=tau, hard=False)
    
    # 计算损失
    rec_loss, ent_loss = loss_function(batch_x, x_recon, probs)
    
    # 组合损失 (熵损失的权重需要根据情况调整)
    total_loss = rec_loss + 0.1 * ent_loss
    
    total_loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Total Loss: {total_loss.item():.6f} | "
              f"MSE: {rec_loss.item():.6f} | Entropy Loss: {ent_loss.item():.6f}")

# ==========================================
# 验证 / 使用
# ==========================================
print("\n训练结束，进行验证...")
model.eval()
with torch.no_grad():
    test_vec = torch.randn(5, D)
    # 推理时使用 Hard=True，这才是真正的 Vector Quantization
    quantized_vec, _, _ = model(test_vec, hard=True)
    
    mse = F.mse_loss(quantized_vec, test_vec)
    print(f"测试集 MSE (Hard Quantization): {mse.item():.6f}")
    
    # 打印原始向量和量化向量对比 (取第一个)
    print("\nOriginal (Top 5 dims):", test_vec[0, :5].numpy())
    print("Quantized(Top 5 dims):", quantized_vec[0, :5].numpy())

开始训练 DPQ... 维度:128, 子空间:4, 码本:256


TypeError: 'int' object is not callable